# LLM Moral Compass — run on Google Colab (free GPU)

**Before anything:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Run the cells from top to bottom (Shift+Enter). Each cell says what it does.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("GPU available:", torch.cuda.is_available())

## 2. Get the code

**Option A (default):** upload the project zip (`llm-moral-compass-v3.zip`) when the button appears.

**Option B:** once the repository is public on GitHub, comment the lines of option A and uncomment the `git clone` line.

In [ ]:
# --- Option A: upload the zip ---
from google.colab import files
uploaded = files.upload()
zip_name = list(uploaded)[0]
!unzip -oq "{zip_name}" -d /content

# --- Option B: clone from GitHub ---
# !git clone https://github.com/corinne3/llm-moral-compass.git /content/llm-moral-compass

%cd /content/llm-moral-compass

## 3. Install the few missing packages and run the tests

In [ ]:
!pip install -q tabulate codecarbon bitsandbytes
!python -m pytest -q

## 4. (Optional) Human data for the moral map

Upload `CountriesChangePr.csv` (see README, section *Human data*). Skip this cell if you don't have it yet.

In [ ]:
from google.colab import files
import os, shutil
up = files.upload()
os.makedirs("data/human", exist_ok=True)
shutil.move(list(up)[0], "data/human/CountriesChangePr.csv")
print("Human file ready.")

## 5. Settings

- `float16` halves the memory on the GPU. If a model gives `mass = nan`, re-run it with `float32`.
- The 0.5B model is included again so that all models are measured in the same conditions.

In [ ]:
# model id -> extra options. The 7B model is loaded in 4 bits to fit the T4 GPU.
MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "",
    "Qwen/Qwen2.5-1.5B-Instruct": "",
    "Qwen/Qwen2.5-3B-Instruct": "",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": "",
    "Qwen/Qwen2.5-7B-Instruct": "--quant 4bit",
}
DTYPE = "float16"
BATCH = 16

## 6. Baseline for every model (a few minutes each)

In [ ]:
for m, extra in MODELS.items():
    print("=" * 30, m)
    !python scripts/01_baseline.py --model {m} --dtype {DTYPE} --batch-size {BATCH} {extra}

## 7. Compare the models: which ones actually read the dilemma?

`order_consistency` is the key: near 0 means the model answers by position, not by content.

In [ ]:
import sys; sys.path.insert(0, "src")
import json, pandas as pd
from moralcompass.cli import slug
rows = []
for m in MODELS:
    d = json.load(open(f"results/{slug(m)}/diagnostics.json"))
    a = pd.read_csv(f"results/{slug(m)}/amce.csv").set_index("dimension")["amce"]
    rows.append({"model": slug(m), **{k: round(v, 3) for k, v in d.items() if k != "n_prompts"},
                 **{f"amce_{k}": round(v, 3) for k, v in a.items()}})
summary = pd.DataFrame(rows).set_index("model")
summary.T

## 8. Moral map (needs the human file)

In [ ]:
!python scripts/02_moral_map.py
from IPython.display import Image, display
import os
if os.path.exists("results/moral_map.png"):
    display(Image("results/moral_map.png"))

## 9. Full study on one model

Choose the model with the best `order_consistency` (and a mass close to 1).

In [ ]:
CHOSEN = "Qwen/Qwen2.5-7B-Instruct"   # <- change after looking at the table above
opts = f"--model {CHOSEN} --dtype {DTYPE} --batch-size {BATCH} {MODELS[CHOSEN]}"
!python scripts/03_persuasion.py {opts}
!python scripts/04_probe_emotions.py {opts}
!python scripts/05_steering.py {opts}
!python scripts/06_side_effects.py {opts}
!python scripts/07_report.py {opts}

## 10. Look at the figures

In [ ]:
from IPython.display import Image, Markdown, display
import glob
from moralcompass.cli import slug
out = f"results/{slug(CHOSEN)}"
for f in sorted(glob.glob(f"{out}/*.png")):
    print(f); display(Image(f))

## 11. Download all the results (zip)

In [ ]:
!zip -qr /content/results.zip results -x "*.pkl"
from google.colab import files
files.download("/content/results.zip")